# Verificación de Entorno — ACA 2026

**Ejecuta este notebook primero antes de comenzar cualquier actividad del curso.**

Verifica que tienes todas las dependencias instaladas y confirma si tienes acceso a GPU.
Si alguna dependencia falla o no tienes GPU, consulta la sección **CPU Fallback** al final.

In [ ]:
# ============================================================
# CELDA 1: Instalar dependencias (solo si ejecutas en Colab)
# ============================================================
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('🔧 Instalando dependencias en Colab...')
    !pip install -q numpy pandas matplotlib seaborn scikit-learn plotly
    !pip install -q torch transformers accelerate
    !pip install -q xgrammar triton
    print('✅ Instalación completa')
else:
    print('ℹ️  Entorno local detectado — se asume que las dependencias ya están instaladas.')

In [ ]:
# ============================================================
# CELDA 2: Versión de Python y plataforma
# ============================================================
import sys, platform
print(f'Python {sys.version}')
print(f'Plataforma: {platform.platform()}')

major, minor = sys.version_info[:2]
if major == 3 and minor >= 9:
    print('✅ Versión de Python correcta (3.9+)')
else:
    print(f'⚠️  Se recomienda Python 3.9+, tienes {major}.{minor}')

In [ ]:
# ============================================================
# CELDA 3: Verificar dependencias de ciencia de datos
# ============================================================
deps_base = {
    'numpy':      '1.24',
    'pandas':     '1.5',
    'matplotlib': '3.6',
    'seaborn':    '0.12',
    'sklearn':    '1.2',
    'plotly':     '5.0',
}

import importlib
all_ok = True
for pkg, min_ver in deps_base.items():
    try:
        # sklearn se importa como scikit-learn pero el módulo es 'sklearn'
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'desconocida')
        print(f'  ✅ {pkg:12s} {ver:10s}  (req: {min_ver}+)')
    except ImportError:
        print(f'  ❌ {pkg:12s} NO INSTALADO   (req: {min_ver}+)')
        all_ok = False

print()
if all_ok:
    print('✅ Todas las dependencias base están presentes.')
else:
    print('❌ Faltan dependencias. Ejecuta: pip install numpy pandas matplotlib seaborn scikit-learn plotly')

In [ ]:
# ============================================================
# CELDA 4: Verificar PyTorch y Transformers
# ============================================================
try:
    import torch
    print(f'  ✅ torch        {torch.__version__}')
except ImportError:
    print('  ❌ torch         NO INSTALADO — pip install torch')

try:
    import transformers
    print(f'  ✅ transformers  {transformers.__version__}')
except ImportError:
    print('  ❌ transformers  NO INSTALADO — pip install transformers')

try:
    import accelerate
    print(f'  ✅ accelerate    {accelerate.__version__}')
except ImportError:
    print('  ⚠️  accelerate   NO INSTALADO (optional) — pip install accelerate')

In [ ]:
# ============================================================
# CELDA 5: Verificar Triton y XGrammar (requeridos para Project_1)
# ============================================================
try:
    import triton
    print(f'  ✅ triton       {triton.__version__}')
except ImportError:
    print('  ⚠️  triton      NO INSTALADO — requerido para módulo Compilers / Project_1')
    print('             pip install triton   (requiere Linux/Mac + CUDA para funcionalidad completa)')
except Exception as e:
    print(f'  ⚠️  triton      instalado pero error al importar: {e}')

try:
    import xgrammar
    print(f'  ✅ xgrammar     {xgrammar.__version__}')
except ImportError:
    print('  ⚠️  xgrammar    NO INSTALADO — requerido para Project_1')
    print('             pip install xgrammar')
except Exception as e:
    print(f'  ⚠️  xgrammar    instalado pero error al importar: {e}')

In [ ]:
# ============================================================
# CELDA 6: Verificar disponibilidad de GPU
# ============================================================
import torch

print('=== GPU / Acelerador ===')
if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    for i in range(n_gpus):
        name = torch.cuda.get_device_name(i)
        mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'  ✅ GPU {i}: {name} ({mem:.1f} GB)')
    print(f'\n  CUDA Version: {torch.version.cuda}')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print('  🍎 Apple MPS disponible (GPU de Apple Silicon)')
    print('  ⚠️  Los kernels Triton NO funcionan en MPS — usa modo CPU Fallback')
else:
    print('  ⚠️  Sin GPU detectada — se usará CPU (más lento)')
    print('  Los notebooks del módulo AI y Stats funcionan bien en CPU.')
    print('  Los notebooks de Project_1 requieren GPU para kernels Triton.')
    print('  → Ver sección CPU Fallback más abajo.')

In [ ]:
# ============================================================
# CELDA 7: Test rápido de funcionalidad básica
# ============================================================
import numpy as np
import torch
import torch.nn.functional as F

# Test numpy
x = np.array([2.0, 1.0, 0.1])
softmax = np.exp(x - x.max()) / np.exp(x - x.max()).sum()
assert abs(softmax.sum() - 1.0) < 1e-6, 'Softmax numpy falló'
print('✅ numpy: softmax OK')

# Test torch
t = torch.tensor([2.0, 1.0, 0.1])
probs = F.softmax(t, dim=-1)
assert abs(probs.sum().item() - 1.0) < 1e-6, 'Softmax torch falló'
print('✅ torch: softmax OK')

# Test matplotlib
import matplotlib
matplotlib.use('Agg')  # no-display backend para test
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
ax.plot([1, 2, 3])
plt.close()
print('✅ matplotlib: plot OK')

print('\n🎉 Entorno básico verificado correctamente.')

---

## 🛟 CPU Fallback

Si **no tienes GPU**, puedes continuar con la mayoría del curso:

| Módulo | ¿Funciona sin GPU? | Notas |
|--------|-------------------|-------|
| AI (L01–L08) | ✅ Sí | Entrenamiento en CPU más lento |
| Compilers | ✅ Sí | Sin dependencia de GPU |
| Stats | ✅ Sí | Sin dependencia de GPU |
| Research | ✅ Sí | Sin dependencia de GPU |
| Project_1 (L01–L05) | ⚠️ Parcial | Los modelos LLM pueden ser lentos |
| Project_1 (L06–L10, kernels Triton) | ❌ Requiere GPU | Usa Google Colab Pro o un servidor con GPU |

### Opciones para acceder a GPU gratuita:

1. **Google Colab** (gratuito con T4): Abre cualquier notebook con el badge [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
2. **Kaggle Kernels** (gratuito con P100): [kaggle.com/code](https://www.kaggle.com/code)
3. **Lightning.ai** (gratuito con A10G limitado): [lightning.ai](https://lightning.ai/)

### Para ejecutar kernels Triton en modo CPU (solo pruebas de lógica):

```bash
TRITON_INTERPRET=1 jupyter notebook
```

Esto ejecuta los kernels Triton en modo interpretado en CPU — no es rendimiento real, pero permite validar que la lógica es correcta.

---

## ✅ Resumen

Si todas las celdas anteriores se ejecutaron sin errores `❌`, estás listo para comenzar.

**Próximo paso recomendado:** Empieza por el [Módulo AI, Lectura 1: IA Clásica vs Generativa](../ai/01_ia_clasica_vs_generativa.md).